# Cerința 4 — Data Pipeline

Un **Data Pipeline** reprezintă un lanț automatizat de transformări care conduc datele dintr-o formă brută la o formă utilizabilă pentru analiză sau modelare.


In [1]:
import os
# Notebook-urile stau în subfolderul notebooks/; ne asigurăm că lucrăm din rădăcina proiectului
# (unde se află data/, models/, plots/), astfel încât toate căile relative să funcționeze.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')
# Directoarele de output trebuie să existe înainte de orice savefig()/save()
os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Working directory:', os.getcwd())

Working directory: /Users/stefan/Documents/football-events-analysis


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnan, floor, year, to_date,
    sum as spark_sum
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, Imputer, Bucketizer
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import os
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('FootballPipeline') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 20:06:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


## 4.1 Pipeline ETL — Citire, Curățare și Feature Engineering

### Etapele pipeline-ului ETL:

```
CSV brut → [1] Citire cu schemă → [2] Filtrare → [3] Curățare null-uri
         → [4] Cast tipuri → [5] Feature engineering → [6] Scriere Parquet
```


In [3]:
# Pasul 1: Citire date brute
print('[ETL] Pasul 1: Citire date brute...')
raw_events = spark.read.csv('data/events.csv', header=True, inferSchema=True, nullValue='NA')
raw_ginf   = spark.read.csv('data/ginf.csv',   header=True, inferSchema=True, nullValue='NA')
print(f'  events: {raw_events.count():,} rânduri | ginf: {raw_ginf.count():,} rânduri')

[ETL] Pasul 1: Citire date brute...


  events: 941,009 rânduri | ginf: 10,112 rânduri


In [4]:
# Pasul 2: Filtrare — păstrăm doar tentativele de șut cu date complete
print('[ETL] Pasul 2: Filtrare tentative de șut...')
shots_raw = raw_events.filter(
    (col('event_type') == 1) &
    col('is_goal').isNotNull() &
    col('location').isNotNull() &
    col('bodypart').isNotNull() &
    col('situation').isNotNull() &
    col('time').isNotNull()
)
print(f'  Rânduri după filtrare: {shots_raw.count():,}')

[ETL] Pasul 2: Filtrare tentative de șut...


  Rânduri după filtrare: 229,135


In [5]:
# Pasul 3: Curățare — înlocuim null-urile din coloane opționale
print('[ETL] Pasul 3: Curățare valori lipsă...')
shots_clean = shots_raw.fillna({
    'assist_method': 0,
    'fast_break':    0,
    'shot_place':    0,
    'side':          1
})

[ETL] Pasul 3: Curățare valori lipsă...


### Feature Engineering — coloane noi derivate

Pe lângă cast-ul explicit la tipuri întregi, construim trei features noi cu `withColumn`: `is_central_shot` (zonele centrale cu conversie mare), `time_phase` (început/mijloc/final de meci) și `is_set_piece` (fază fixă). Modelul primește astfel semnale pe care nu le-ar putea deduce ușor din codurile brute.


In [6]:
# Pasul 4 & 5: Cast tipuri corecte + Feature Engineering
print('[ETL] Pasul 4-5: Cast tipuri + Feature Engineering...')
shots_engineered = shots_clean \
    .withColumn('is_goal',      col('is_goal').cast(IntegerType())) \
    .withColumn('location',     col('location').cast(IntegerType())) \
    .withColumn('bodypart',     col('bodypart').cast(IntegerType())) \
    .withColumn('situation',    col('situation').cast(IntegerType())) \
    .withColumn('assist_method',col('assist_method').cast(IntegerType())) \
    .withColumn('fast_break',   col('fast_break').cast(IntegerType())) \
    .withColumn('shot_place',   col('shot_place').cast(IntegerType())) \
    .withColumn('time',         col('time').cast(DoubleType())) \
    .withColumn('side',         col('side').cast(IntegerType())) \
    \
    .withColumn('is_central_shot',
                when(col('location').isin(3, 5, 10, 12, 13, 14), 1).otherwise(0)) \
    \
    .withColumn('time_phase',
                when(col('time') <= 45, 1)           # prima repriză
                .when(col('time') <= 70, 2)           # a doua repriză - început
                .otherwise(3)) \
    \
    .withColumn('is_set_piece',
                when(col('situation').isin(2, 3, 4), 1).otherwise(0)) \
    \
    .select('id_odsp', 'id_event', 'is_goal',
            'location', 'bodypart', 'situation', 'assist_method',
            'fast_break', 'shot_place', 'side', 'time',
            'is_central_shot', 'time_phase', 'is_set_piece')

print(f'  Coloane după feature engineering: {len(shots_engineered.columns)}')
shots_engineered.show(3)

[ETL] Pasul 4-5: Cast tipuri + Feature Engineering...
  Coloane după feature engineering: 14


+---------+----------+-------+--------+--------+---------+-------------+----------+----------+----+----+---------------+----------+------------+
|  id_odsp|  id_event|is_goal|location|bodypart|situation|assist_method|fast_break|shot_place|side|time|is_central_shot|time_phase|is_set_piece|
+---------+----------+-------+--------+--------+---------+-------------+----------+----------+----+----+---------------+----------+------------+
|UFot0hit/| UFot0hit1|      0|       9|       2|        1|            1|         0|         6|   2| 2.0|              0|         1|           0|
|UFot0hit/|UFot0hit12|      0|      15|       1|        1|            1|         0|        13|   1|14.0|              0|         1|           0|
|UFot0hit/|UFot0hit14|      1|       9|       2|        1|            1|         0|         4|   1|17.0|              0|         1|           0|
+---------+----------+-------+--------+--------+---------+-------------+----------+----------+----+----+---------------+----------

### Scrierea în Parquet partiționat — capătul pipeline-ului ETL

Parquet e format *columnar*: compresie mai bună (valori similare adiacente), citire doar a coloanelor cerute (column pruning) și filtrare la citire (predicate pushdown). `partitionBy('situation')` împarte fizic datele în subdirectoare (`situation=1/`, `situation=2/`...) — un filtru ulterior pe `situation` va sări complet peste partițiile irelevante (partition pruning).


In [7]:
# Pasul 6: Scriere în format Parquet (stocare optimizată)
print('[ETL] Pasul 6: Salvare în format Parquet...')
parquet_path = 'data/processed_shots'

shots_engineered.write \
    .mode('overwrite') \
    .partitionBy('situation') \
    .parquet(parquet_path)

print(f'  Salvat în: {parquet_path}/')

# Verificare citire Parquet
shots_parquet = spark.read.parquet(parquet_path)
print(f'  Citit din Parquet: {shots_parquet.count():,} rânduri')
print(f'  Schema Parquet:')
shots_parquet.printSchema()

[ETL] Pasul 6: Salvare în format Parquet...


  Salvat în: data/processed_shots/
  Citit din Parquet: 229,135 rânduri
  Schema Parquet:
root
 |-- id_odsp: string (nullable = true)
 |-- id_event: string (nullable = true)
 |-- is_goal: integer (nullable = true)
 |-- location: integer (nullable = true)
 |-- bodypart: integer (nullable = true)
 |-- assist_method: integer (nullable = true)
 |-- fast_break: integer (nullable = true)
 |-- shot_place: integer (nullable = true)
 |-- side: integer (nullable = true)
 |-- time: double (nullable = true)
 |-- is_central_shot: integer (nullable = true)
 |-- time_phase: integer (nullable = true)
 |-- is_set_piece: integer (nullable = true)
 |-- situation: integer (nullable = true)



## 4.2 Pipeline MLlib — Preprocesare + Model ML

Construim un **Pipeline MLlib** complet care:
1. **Imputează** valorile lipsă rămase
2. **Asamblează** toate features într-un singur vector
3. **Scalează** features numerice (StandardScaler)
4. **Antrenează** un Random Forest Classifier
5. **Serializează** întregul pipeline pe disc — poate fi reîncărcat și aplicat pe date noi fără reantrenare


In [8]:
# Citim datele procesate din Parquet
data_df = spark.read.parquet('data/processed_shots')

feature_cols = ['location', 'bodypart', 'situation', 'assist_method',
                'fast_break', 'shot_place', 'side', 'time',
                'is_central_shot', 'time_phase', 'is_set_piece']

# Etapa 1: Imputare (înlocuire null cu mediană)
imputer = Imputer(
    inputCols=feature_cols,
    outputCols=[f'{c}_imp' for c in feature_cols],
    strategy='median'
)

# Etapa 2: VectorAssembler
assembler = VectorAssembler(
    inputCols=[f'{c}_imp' for c in feature_cols],
    outputCol='features_raw',
    handleInvalid='skip'
)

# Etapa 3: StandardScaler
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True
)

# Etapa 4: Model RF
rf_model = RandomForestClassifier(
    labelCol='is_goal',
    featuresCol='features',
    numTrees=100,
    maxDepth=8,
    seed=42
)

# Construim Pipeline-ul complet
full_pipeline = Pipeline(stages=[imputer, assembler, scaler, rf_model])

print('Structura Pipeline-ului MLlib:')
for i, stage in enumerate(full_pipeline.getStages()):
    print(f'  Etapa {i+1}: {type(stage).__name__}')

Structura Pipeline-ului MLlib:
  Etapa 1: Imputer
  Etapa 2: VectorAssembler
  Etapa 3: StandardScaler
  Etapa 4: RandomForestClassifier


### Antrenarea și evaluarea pipeline-ului complet

`full_pipeline.fit(train_df)` rulează etapele în ordine: estimatorii (`Imputer`, `StandardScaler`, `RandomForestClassifier`) învață din date și devin transformeri; transformerii (`VectorAssembler`) se aplică direct. Rezultatul e un singur obiect `PipelineModel` care încapsulează tot lanțul de preprocesare + model.


In [9]:
train_df, test_df = data_df.randomSplit([0.8, 0.2], seed=42)

print('Antrenare Pipeline MLlib complet...')
pipeline_model = full_pipeline.fit(train_df)
print('Pipeline antrenat.')

predictions = pipeline_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol='is_goal', metricName='areaUnderROC')
auc = evaluator.evaluate(predictions)
print(f'\nAUC-ROC pe test set: {auc:.4f}')

Antrenare Pipeline MLlib complet...


Pipeline antrenat.



AUC-ROC pe test set: 0.9433


### Serializarea și reîncărcarea — dovada reproductibilității

Salvăm pipeline-ul întreg (nu doar modelul) pe disc și îl reîncărcăm cu `PipelineModel.load`. AUC-ul identic înainte/după demonstrează că artefactul conține *tot* ce e nevoie la inferență: mediana imputer-ului, media/deviația scaler-ului și toți arborii RF.


In [10]:
# Serializarea Pipeline-ului antrenat pe disc
pipeline_save_path = 'models/rf_pipeline_model'
os.makedirs('models', exist_ok=True)

pipeline_model.write().overwrite().save(pipeline_save_path)
print(f'Pipeline salvat în: {pipeline_save_path}')

# Reîncărcăm pipeline-ul pentru a demonstra reproductibilitatea
from pyspark.ml import PipelineModel
loaded_pipeline = PipelineModel.load(pipeline_save_path)

preds_loaded = loaded_pipeline.transform(test_df)
auc_loaded   = evaluator.evaluate(preds_loaded)
print(f'AUC-ROC pipeline reîncărcat: {auc_loaded:.4f} (trebuie să fie identic cu {auc:.4f})')

spark.stop()
print('\nSesiune Spark oprită.')

Pipeline salvat în: models/rf_pipeline_model


AUC-ROC pipeline reîncărcat: 0.9433 (trebuie să fie identic cu 0.9433)



Sesiune Spark oprită.
